# H&E 组织掩膜提取：使用与调参手册

配套脚本：`he_tissue_mask.py`。下面所有执行单元格都用 `!python 脚本绝对路径 ...` 的方式调用，
把命令中的三个路径换成自己的即可（示例用的是本项目 PGD 数据集的真实路径）。

**推荐顺序：安装环境 → 批量提取 → 检查预览 → 单张调参 → 保存配置。**

脚本以 TIAToolbox 的 Otsu 为基础；`enhanced` 模式额外使用自定义背景校正、光密度和饱和度补充。跨批次效果需要实际图像验证。

## 1. 安装和选择 Jupyter 内核

在终端执行一次：

```bash
conda create -n he_mask python=3.11 -y
conda activate he_mask
pip install tiatoolbox==2.1.3 ipykernel jupyterlab
python -m ipykernel install --user --name he_mask --display-name "Python (he_mask)"
jupyter lab
```

打开 Notebook 后选择 **Python (he_mask)** 内核。激活终端中的 conda 环境不会自动切换已打开的 Notebook 内核。

`!python` 使用的是当前内核所在环境的 Python，所以内核必须选对；不确定时运行下一格自检。

In [ ]:
import sys
import importlib.metadata
print("当前 Python：", sys.executable)
try:
    print("TIAToolbox：", importlib.metadata.version("tiatoolbox"))
    from tiatoolbox.tools.tissuemask import OtsuTissueMasker
    print("组织掩膜模块导入成功")
except Exception as exc:
    print("请检查环境及依赖：", exc)

## 2. 两种掩膜模式

用 `--mask-mode` 选择，默认 `fine`。两种模式的阈值分割完全一样，区别只在形态学后处理。

| 模式 | 命令 | 结果 |
|---|---|---|
| 精细 `fine` | 默认，可写 `--mask-mode fine` | 贴合真实染色分布，组织内部保留腔隙、空洞和碎点 |
| 粗糙 `coarse` | `--mask-mode coarse` | 只保留组织外轮廓，内部空洞全部填平，边界更平滑 |

`coarse` 的处理顺序：闭运算桥接组织内部缝隙 → 填掉全部内部空洞 → 开运算平滑轮廓、去掉毛刺 → 再填一次空洞 → 按面积删除小碎块。
只有和图像边界相连的背景不会被填，所以组织外侧的真实背景不会被误填。

| 粗糙模式参数 | 默认 | 作用 |
|---|---|---|
| `coarse_close_radius` | 6 | 桥接组织内部缝隙的半径；调大可闭合更宽的裂缝，但可能把相邻两块组织连成一块 |
| `coarse_fill_holes` | true | 是否填掉全部内部空洞；设为 false 就只做闭运算和开运算 |
| `coarse_open_radius` | 2 | 平滑轮廓、去掉细长毛刺；调大会磨掉细小突起 |
| `coarse_min_area` | 2000 | 填洞后仍小于该面积（工作图像像素数）的连通块会被删除 |
| `coarse_max_hole_frac` | 0.5 | 安全阀：面积超过整图该比例的“空洞”不填。防止掩膜沿图像四周形成闭环时把整张图填成组织 |

半径单位是**工作图像像素**，面积单位是**工作图像像素数**；改了 `max_dim` 或 `work_mpp` 之后要重新检查这几个值。

另外脚本在读图后会自动修掉缩略图读取器在图像最外圈产生的 1 像素黑边（TIAToolbox 生成缩略图时，滤波窗口越界的部分按黑色填充，最外圈亮度只有相邻行的一半）。
这圈黑边比任何真实背景都暗，Otsu 一定判成组织，于是掩膜沿四周形成闭环，粗糙模式填洞时会把整张图填满。修复由 `edge_artifact_ratio`（默认 0.8）控制：
某条边的平均亮度低于相邻一行/一列的 0.8 倍时，用相邻行列替换它，并在 JSON 的 `warnings` 里记一条 `Replaced dark 1-px reader border`。设为 0 可关闭该修复。

## 3. 输出结构

`--mask-dir` 下固定生成三个子文件夹，文件名统一用输入文件的主名（stem）：

```
mask_dir/
├── json/      S004_5.json          参数、尺寸、坐标尺度、诊断信息
│              S004_5.error.json    仅在该图处理失败时出现
├── mask/      S004_5.tif           单通道二值掩膜 TIFF，背景 0、组织 255
└── preview/   S004_5.jpg           原图 | 掩膜 | 绿色叠加 三联图
```

带 `--recursive` 时，输入的子目录层级会在这三个文件夹下各自复现，例如 `mask/batch1/section01.tif`。

因为输出按主名命名，**同一目录下不允许出现 `S004_5.tif` 和 `S004_5.png` 这种同名不同扩展名的输入**，脚本会在开始处理前直接报错并列出冲突文件。

掩膜 TIFF 用 deflate 压缩，`tifffile.imread`、`cv2.imread(..., IMREAD_UNCHANGED)`、`PIL.Image.open` 都能直接读。

## 4. 批量提取

首次使用默认参数。`json/`、`mask/`、`preview/` 三个文件都已存在时会跳过该图；重新生成需要 `--overwrite`。

默认最长边不超过 **3000 像素**，不放大较小的输入。每张图单独估计阈值，输出掩膜通常不是原图尺寸。

输入有子文件夹时加 `--recursive`。输出目录必须位于输入目录之外。

In [ ]:
# 精细模式（默认）批量提取
!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --HE-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image \
    --mask-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA

In [ ]:
# 或者这种调用方式，
# !python —— 那调用的是启动 Jupyter 时 shell 里的 conda 环境
# 用 sys.executable 才能保证使用jupyter的内核环境。
import sys
import subprocess
from pathlib import Path

HE_DIR = Path("/mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image")
MASK_DIR = Path("/mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA_1")
SCRIPT = Path("/mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py")

print("kernel python:", sys.executable)

cmd = [
    sys.executable,
    str(SCRIPT),
    "--HE-dir", str(HE_DIR),
    "--mask-dir", str(MASK_DIR),
    "--mask-mode", "coarse",
    "--full-size",
    # 默认上限 1e8 像素；超限会报 Full-size mask exceeds max-full-pixels
    "--max-full-pixels", "500000000",
    "--overwrite",
]
print("cmd:", " ".join(cmd))
subprocess.check_call(cmd)

粗糙模式建议输出到另一个目录，方便和精细模式对比，也避免互相覆盖。

In [ ]:
# 粗糙模式：只保留组织外轮廓
!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --HE-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image \
    --mask-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA_coarse \
    --mask-mode coarse

## 5. 在 Notebook 中检查预览和参数

先完成提取，再运行下面的单元格。改 `NAME` 可以查看别的图；有子目录时写成 `batch1/section01`（不带扩展名）。

低于 1% 或高于 95% 的组织比例会触发脚本提示；这只是复核提示，并不自动意味着结果错误。
`diagnostics` 里的 `tissue_fraction` 是最终掩膜的比例，`fine_tissue_fraction` 是填洞之前的比例，两者之差就是粗糙模式填掉的面积。

In [ ]:
import json
from pathlib import Path
from IPython.display import display, Image as DisplayImage

MASK_DIR = Path("/mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA")
NAME = "S004_5"   # 输入文件主名，不带扩展名

preview_path = MASK_DIR / "preview" / f"{NAME}.jpg"
meta_path = MASK_DIR / "json" / f"{NAME}.json"
error_path = MASK_DIR / "json" / f"{NAME}.error.json"

if preview_path.exists():
    display(DisplayImage(filename=str(preview_path), width=1200))
else:
    print("预览不存在：", preview_path)
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    print(json.dumps({k: meta[k] for k in ("parameters", "diagnostics", "original_wh", "mask_wh")},
                     ensure_ascii=False, indent=2))
if error_path.exists():
    print("该图处理失败：", error_path.read_text(encoding="utf-8"))

并排比较两种模式的叠加预览：

In [ ]:
from IPython.display import display, Markdown, Image as DisplayImage

FINE_DIR = Path("/mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA")
COARSE_DIR = Path("/mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA_coarse")

for label, d in [("精细 fine", FINE_DIR), ("粗糙 coarse", COARSE_DIR)]:
    path = d / "preview" / f"{NAME}.jpg"
    display(Markdown(f"**{label}**：{path}"))
    if path.exists():
        display(DisplayImage(filename=str(path), width=1100))
    else:
        print("不存在，请先运行对应模式的批量提取")

## 6. 单张调参重跑

`--only` 填**相对输入目录的完整文件名**（带扩展名，子目录用 `/` 分隔，例如 `batch1/section01.tif`）。
重跑同一张图必须加 `--overwrite`，否则会被跳过。

先调整 `sensitivity`。其默认值为 1.0：增大通常减少淡染漏检，减小通常减少背景误检。不要一开始同时修改很多参数。

| 问题 | 调整示例 |
|---|---|
| 淡染组织漏检 | `sensitivity=1.15` 或 `1.3` |
| 背景误识别 | `sensitivity=0.85` |
| 小碎点太多 | `min_area=500` |
| 真实小组织块丢失 | `min_area=30` |
| 组织内部小孔太多（精细模式） | `hole_area=300` |
| 不进行显式孔洞填充 | `hole_area=0` |
| 细小断裂需要连接 | `close_radius=2` |
| 边缘需要略微扩张 | `dilate_radius=1` |
| 确定只需要最大组织块 | `keep_largest=1` |

面积参数单位为**工作图像像素数**，半径为**工作图像像素**。改变工作分辨率后需要重新检查。

`hole_area=0` 仅关闭显式填孔；闭运算仍可能封闭小孔或窄缝。若希望减少这类改变，同时使用 `close_radius=0`、`dilate_radius=0`。
默认 `keep_largest=0` 保留所有通过面积筛选的组织块。

In [ ]:
# 淡染漏检：只重跑指定的一张图
!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --HE-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image \
    --mask-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA \
    --only S004_5.tif \
    --set sensitivity=1.2 \
    --overwrite

In [ ]:
# 背景误检或碎点较多：根据实际情况选用
!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --HE-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image \
    --mask-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA \
    --only S004_5.tif \
    --set sensitivity=0.85 \
    --set min_area=500 \
    --overwrite

### 粗糙模式的单张调参

轮廓上还留着没填掉的缺口，通常是组织边缘有裂缝、空洞和外部背景连通，填洞算法按定义不会填这种缺口。
这时调大 `coarse_close_radius` 先把裂缝闭合；调得过大会把相邻组织块粘连，需要看预览确认。

In [ ]:
# 裂缝没闭合：加大闭运算半径，并放宽小碎块阈值
!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --HE-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image \
    --mask-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA_coarse \
    --mask-mode coarse \
    --only S004_5.tif \
    --set coarse_close_radius=12 \
    --set coarse_min_area=4000 \
    --overwrite

### 完全手动指定灰度阈值

切换 `mode=otsu`，关闭光密度和饱和度补充。组织条件为 `gray < gray_threshold`，阈值越大通常保留越多。

注意 `mode` 和 `mask_mode` 是两个不同的参数：`mode` 决定怎么分割（`enhanced` / `otsu`），`mask_mode` 决定分割之后怎么后处理（`fine` / `coarse`），两者可以任意组合。

阈值作用于背景校正、平滑后的图像。希望在未经这两步处理的 RGB 灰度上调阈值，可额外设置 `background_correct=false` 和 `blur_sigma=0`。
`gray_threshold` 显式给定时，灰度阈值不再由 sensitivity 推导。

In [ ]:
!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --HE-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image \
    --mask-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA \
    --only S004_5.tif \
    --set mode=otsu \
    --set gray_threshold=225 \
    --overwrite

### 可选：分目录比较不同敏感度

不同参数结果存入不同试验目录，检查叠加预览后再把选定参数用于正式输出。

In [ ]:
!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --HE-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image \
    --mask-dir /tmp/he_mask_trial_s0.85 --only S004_5.tif --set sensitivity=0.85 --overwrite

!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --HE-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image \
    --mask-dir /tmp/he_mask_trial_s1.2 --only S004_5.tif --set sensitivity=1.2 --overwrite

## 7. 持久化全局与单张参数

配置分为 `defaults` 和 `overrides`；单张键为**精确的相对路径**（带扩展名，与 `--only` 写法一致）。

**优先级：命令行 `--mask-mode` / `--set` > 单张 overrides > 配置 defaults > 脚本内置默认值。**

下面的单元格会创建配置 JSON；文件已存在时 `--init-config` 会直接报错，不会覆盖。初始化模板里带一个 `section01.tif` 示例覆盖项，使用前请按实际文件调整或删除。

In [ ]:
!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --init-config /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_mask_config.json

编辑下方字典后运行即可写入配置。先以 `SAVE_CONFIG=False` 检查内容，再改为 `True` 保存；保存前会备份已有 JSON。
此示例保留已有其他键，并更新指定两张图的覆盖项（其中一张单独使用粗糙模式）。

In [ ]:
import json
import shutil
from datetime import datetime
from pathlib import Path

CONFIG_PATH = Path("/mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_mask_config.json")

config = json.loads(CONFIG_PATH.read_text(encoding="utf-8")) if CONFIG_PATH.exists() else {}
config.setdefault("defaults", {})
config.setdefault("overrides", {}).update({
    "S004_5.tif": {"sensitivity": 1.2, "min_area": 50},
    "S003_1.tif": {"mask_mode": "coarse", "coarse_close_radius": 12},
})
print(json.dumps(config, ensure_ascii=False, indent=2))

SAVE_CONFIG = False
if SAVE_CONFIG:
    if CONFIG_PATH.exists():
        stamp = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
        shutil.copy2(CONFIG_PATH, CONFIG_PATH.with_name(CONFIG_PATH.name + f".{stamp}.bak"))
    CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    print("已保存：", CONFIG_PATH)

In [ ]:
# 采用配置批量重跑；也可加 --only 仅处理单张
!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --HE-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image \
    --mask-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA \
    --config /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_mask_config.json \
    --overwrite

## 8. 分辨率与原图尺寸输出

`--full-size` 用最近邻放大工作掩膜，保持 0/255，**不会增加组织边缘细节**。默认限制一亿像素，避免导出超大掩膜；超限会报错而非静默降级。
如果确实需要更精细的边缘，应先提高 `max_dim`，同时重新检查形态学参数（包括粗糙模式那几个）。

In [ ]:
!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --HE-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image \
    --mask-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA_fullsize \
    --full-size --overwrite

### 不同批次统一工作尺度

有可靠 MPP 的 WSI 可设置 `work_mpp=8`（8 µm/像素）。PNG/JPG 没有可靠物理元数据时，还需填写真实 `input_mpp`。

`max_dim` 仍然约束最大图像尺寸；实际工作 MPP 可能比请求值更粗。以每张 JSON 的 `working_mpp_xy` 为准。脚本不放大原始低分辨率输入。

以下 `input_mpp=0.5` **只是示例，必须改为真实值**。对于缩略图或重采样图像，应填写该输入图本身的 MPP，不是扫描原片的 MPP。
本项目 `downsample_image` 是按 0.5 µm/pixel 再降采样得到的，填写前请核对实际比例。

In [ ]:
# WSI 已有可靠 MPP
!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --HE-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image \
    --mask-dir /tmp/he_mask_mpp8 --set work_mpp=8 --overwrite

# 普通 RGB 图像：0.5 必须替换成该输入图像真实的 µm/像素
!python /mnt/zzf_nas/A_xff/Spaceland-omics/spaceland_omics/data_preprocess/he_tissue_mask.py \
    --HE-dir /mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image \
    --mask-dir /tmp/he_mask_mpp8 --set work_mpp=8 --set input_mpp=0.5 --overwrite

## 9. 读取二值掩膜及其坐标尺度

数组下标为 `[row, col]`，坐标为 `(x, y)=(col, row)`。JSON 的尺寸顺序为 `[width, height]`。

低分辨率掩膜覆盖完整输入视野，不裁剪、不旋转。以图像左上边界为原点时，掩膜像素 `(col,row)` 中心在原图连续边界坐标中为 `((col+0.5)*sx, (row+0.5)*sy)`。
若使用以左上像素中心为零点的坐标约定，再减去 0.5。进行 spot 配准或坐标转换时应保持约定一致。

In [ ]:
import json
import numpy as np
from pathlib import Path
from PIL import Image

MASK_DIR = Path("/mnt/zzf_nas/A_xff/Spaceland-omics/data/interim/PGD_dataset/HE_unregistered/downsample_image_TA")
NAME = "S004_5"

mask_path = MASK_DIR / "mask" / f"{NAME}.tif"
meta_path = MASK_DIR / "json" / f"{NAME}.json"
if mask_path.exists() and meta_path.exists():
    mask = np.asarray(Image.open(mask_path)) > 0
    meta = json.loads(meta_path.read_text(encoding="utf-8"))
    sx, sy = meta["baseline_pixels_per_mask_pixel_xy"]
    print("mask shape (H,W)：", mask.shape, " 组织像素占比：", f"{mask.mean():.1%}")
    print("原图像素 / 掩膜像素 (x,y)：", sx, sy)
    print("实际工作 MPP：", meta["working_mpp_xy"])
    print("掩膜模式：", meta["parameters"]["mask_mode"])
else:
    print("请先生成掩膜和 JSON")

## 10. 常见问题与验证范围

| 情况 | 检查方法 |
|---|---|
| 导入报错 | 确认 Jupyter 内核及 TIAToolbox 完整依赖 |
| 修改参数后没有变化 | 添加 `--overwrite`；确认是否有更高优先级的参数 |
| `No matching images` | 检查扩展名、`--only` 的相对路径和是否需要 `--recursive` |
| `Inputs share an output name` | 同一目录下有同主名不同扩展名的输入，重命名后再跑 |
| 粗糙模式整张图全白（`tissue_fraction=1.000`） | 掩膜沿图像四周形成了闭环。检查 JSON 的 `warnings`：有 `Skipped filling a hole...` 说明安全阀已拦住，但环状伪影仍在，需查明来源；`edge_artifact_ratio` 被改成 0 时会复现该问题 |
| 粗糙模式仍有缺口 | 缺口与外部背景连通，调大 `coarse_close_radius` |
| 粗糙模式把两块组织粘连 | 调小 `coarse_close_radius`，或改用精细模式 |
| 粗糙模式丢掉了小组织块 | 调小 `coarse_min_area` |
| `work_mpp` 缺少元数据 | 指定真实 `input_mpp`，或使用默认 `max_dim` 工作方式 |
| 掩膜和原图尺寸不同 | 默认行为；使用 JSON 中的比例或加 `--full-size` |
| 特定图像处理失败 | 查看 `json/<主名>.error.json`；其他图像仍会继续处理 |
| 背景估计不可靠 | 检查预览，可设置 `background_rgb`，例如 `--set background_rgb=[245,243,242]`，数值需从真实玻片背景估计 |
| 大量笔迹、折叠或严重照明不均 | 阈值法可能无法区分伪影与组织，需要针对性处理或其他分割方法 |

当前脚本要求 8-bit RGB/RGBA H&E；其他位深或通道请显式预处理。TIFF/WSI 的支持情况取决于 TIAToolbox 的 reader 和文件编码。

**验证范围**：两种模式、三文件夹输出、同名冲突检测、`--recursive` 与跳过逻辑已在本项目 `downsample_image` 的样本上实际跑通；
未逐张复核全部切片的分割质量，也未验证其他批次样本。调参单元格给出的数值均为示例，需要按预览结果确认。

官方参考：
- [TIAToolbox 组织掩膜源码](https://tia-toolbox.readthedocs.io/en/latest/_modules/tiatoolbox/tools/tissuemask.html)
- [组织掩膜教程](https://tia-toolbox.readthedocs.io/en/latest/_notebooks/jnb/03-tissue-masking.html)
- [WSI 读取与分辨率](https://tia-toolbox.readthedocs.io/en/v2.0.0/_notebooks/jnb/01-wsi-reading.html)